# Modelado BTS — Etapa A: ingeniería de variables por lotes (AWS)

Este notebook conserva la preparación y cálculos de variables originales, las fechas y los tamaños de muestra (`4 000 000 / 1 000 000 / 1 000 000`), pero **no intenta mantener los tres DataFrames completos en memoria**.

La muestra de cada tramo se selecciona una sola vez mediante el mismo `USING SAMPLE reservoir(...) REPEATABLE (SEED)` usado en el notebook original. Una tabla de muestra se guarda en DuckDB; se calculan las variables en **bloques de filas de esa misma muestra**, utilizando las tablas auxiliares de todo Silver. Los bloques se guardan como Parquet para etapas posteriores.

**Nota:** requiere espacio libre en EBS tanto para la base DuckDB persistente como para los Parquet y archivos temporales. Si el notebook falla durante la creación de tablas auxiliares, hay que tratar esa etapa aparte. No garantiza por sí solo que LightGBM o XGBoost entrenen con 4 millones de filas dentro de 16 GiB: para eso se ejecutarán los modelos por separado.


In [3]:
%pip install -q duckdb lightgbm xgboost pyarrow boto3 scikit-learn matplotlib pandas


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.cluster import MiniBatchKMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             log_loss, precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [3]:
SILVER_GLOB = "../data/silver/flights/**/*.parquet"
OUT_DIR     = Path("../outputs/outputs_modeling")
TARGET      = "DepDel15"
SEED        = 42

MEMORY_LIMIT = "6GB"
TEMP_DIR     = "duckdb_tmp"

# --- Partición cronológica (fechas inclusivas). Prueba = todo lo posterior a VAL_END. ---
TRAIN_END   = "2024-12-31"
VAL_END     = "2025-08-31"
WARMUP_DAYS = 60            # los primeros días no tienen historial: no se usan para entrenar

# --- Tamaño de las muestras (None = todo el tramo; con 20 M de filas necesitas bastante RAM) ---
N_TRAIN   = 4_000_000
N_EVAL    = 1_000_000       # validación y prueba
N_LOGREG  = 1_000_000       # filas de entrenamiento de la regresión logística
N_KMEANS  = 1_000_000       # filas con las que se ajusta K-means

# --- Patrones históricos y estado del día ---
HIST_WINDOW_DAYS = 30
HIST_SMOOTH      = 50
PRED_HORIZON_MIN = 240      # se predice 240 min antes de la salida programada: el estado del día
                            # (retrasos de vuelos anteriores) solo usa lo ya ocurrido a esa hora

# --- Costos del umbral: costo de un retraso no detectado vs. el de una falsa alarma ---
COST_FP = 1
COST_FN = 3                 # supuesto: perder un retraso cuesta 3 veces una falsa alarma

# --- Modelos ---
LGB_MAX_TREES  = 3000
LGB_EARLY_STOP = 100
K_GRID         = [10, 25, 50, 100]

# --- Significancia (bootstrap pareado sobre una submuestra de prueba) ---
BOOT_ROWS = 300_000
BOOT_N    = 100

# --- Secciones opcionales ---
RUN_ABLATION = True         # LightGBM con distintos grupos de variables
RUN_WEIGHTED = True         # experimento de pesos de clase
RUN_DIAG     = True         # diagnóstico de deriva (entrena 2022-2023, evalúa 2024)
RUN_REFIT    = True         # reentrena el ganador con entrenamiento + validación
N_DIAG_TRAIN = 1_500_000
N_DIAG_EVAL  = 500_000

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
Path(TEMP_DIR).mkdir(exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

In [4]:
# ===== AWS: trae la capa silver (y el gold previo, si existe) de S3 al disco local =====
# Así el resto del cuaderno corre SIN cambios: sigue leyendo ../data/silver/flights/**/*.parquet.
import os
import boto3
from botocore.exceptions import ClientError

BUCKET         = os.environ.get("BTS_BUCKET", "bts-flights-analysis")   # <- tu bucket
S3_GOLD_PREFIX = "gold/pred_vs_real/"            # una carpeta por tabla (Athena necesita eso)
S3_OUT_PREFIX  = "modeling/outputs_modeling/"    # reportes, figuras y modelo
s3 = boto3.client("s3")

def download_prefix(prefix, dest, suffix=".parquet"):
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            if not obj["Key"].endswith(suffix):
                continue                          # ignora _SUCCESS y otros marcadores de Spark
            target = Path(dest) / obj["Key"][len(prefix):]
            target.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(BUCKET, obj["Key"], str(target))
            n += 1
    return n

n_silver = download_prefix("silver/flights/", "../data/silver/flights")
assert n_silver > 0, f"No hay archivos en s3://{BUCKET}/silver/flights/"
print(f"Silver descargado: {n_silver} archivos parquet desde s3://{BUCKET}/silver/flights/")

# Gold previo: write_gold() conserva otros conjuntos (p. ej. 'prospectivo'), así que hay que traerlo antes.
GOLD_LOCAL = Path("../data/gold/pred_vs_real"); GOLD_LOCAL.mkdir(parents=True, exist_ok=True)
for _name in ["group_metrics", "score_bins", "timeseries", "heat_hora_dia"]:
    try:
        s3.download_file(BUCKET, f"{S3_GOLD_PREFIX}{_name}/{_name}.parquet", str(GOLD_LOCAL / f"{_name}.parquet"))
        print("  gold previo:", _name)
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
            raise
try:
    s3.download_file(BUCKET, f"{S3_GOLD_PREFIX}meta.json", str(GOLD_LOCAL / "meta.json"))
except ClientError as e:
    if e.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
        raise

Silver descargado: 56 archivos parquet desde s3://bts-flights-analysis/silver/flights/


## 1. Preparar DuckDB persistente y auxiliares
Las vistas y tablas auxiliares se crean en un archivo de DuckDB que persiste tras reiniciar el kernel. **No elimines el archivo de la base de datos mientras está en uso.**

In [5]:
DB_PATH = Path("../data/checkpoints/modeling.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH))
con.execute("SET threads = 2")
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET temp_directory = '{TEMP_DIR}'")
con.execute(f"""
    CREATE OR REPLACE VIEW silver AS
    SELECT * FROM read_parquet('{SILVER_GLOB}', hive_partitioning = true,
                               hive_types = {{'Year': 'SMALLINT', 'Month': 'TINYINT'}})
""")

n_rows, d_min, d_max = con.sql("SELECT COUNT(*), MIN(FlightDate), MAX(FlightDate) FROM silver").fetchall()[0]
print(f"Silver: {n_rows:,} filas | {d_min} → {d_max}")
assert con.sql(f"SELECT COUNT(*) - COUNT({TARGET}) FROM silver").fetchall()[0][0] == 0, "Hay objetivos nulos en silver"

SPLIT_SQL = (f"CASE WHEN FlightDate <= DATE '{TRAIN_END}' THEN 'train' "
             f"WHEN FlightDate <= DATE '{VAL_END}' THEN 'val' ELSE 'test' END")
split_summary = con.sql(f"""
    SELECT {SPLIT_SQL} AS split, MIN(FlightDate) AS desde, MAX(FlightDate) AS hasta,
           COUNT(*) AS vuelos, AVG({TARGET}) AS tasa_retraso
    FROM silver GROUP BY 1 ORDER BY CASE split WHEN 'train' THEN 0 WHEN 'val' THEN 1 ELSE 2 END
""").df()
split_summary

Silver: 31,833,006 filas | 2022-01-01 → 2026-08-31


,split,desde,hasta,vuelos,tasa_retraso
0,train,2022-01-01,2024-12-31,20301229,0.2084
1,val,2025-01-01,2025-08-31,4610259,0.2226
2,test,2025-09-01,2026-08-31,6921518,0.2294


In [6]:
# Calendario con la distancia (en días) al feriado federal más cercano, acotada a 10.
cal_dates = pd.date_range(pd.Timestamp(d_min) - pd.Timedelta(days=15), pd.Timestamp(d_max) + pd.Timedelta(days=15))
hol = USFederalHolidayCalendar().holidays(start=cal_dates.min(), end=cal_dates.max()).values.astype("datetime64[D]")
cal_np = cal_dates.values.astype("datetime64[D]")
days_to_hol = np.abs(cal_np[:, None] - hol[None, :]).min(axis=1).astype(float).clip(max=10)
cal_df = pd.DataFrame({"FlightDate": cal_dates.date, "days_to_holiday": days_to_hol})
con.register("cal_df", cal_df)
con.execute("CREATE OR REPLACE TABLE calendar AS SELECT CAST(FlightDate AS DATE) AS FlightDate, days_to_holiday FROM cal_df")

con.execute(f"""
    CREATE OR REPLACE VIEW base AS
    SELECT
        FlightDate,
        CAST({TARGET} AS INTEGER)                              AS {TARGET},
        Reporting_Airline, Origin, Dest, OriginState, DestState, DepTimeBlk,
        Origin || '-' || Dest                                  AS route,
        NULLIF(TRIM(Tail_Number), '')                          AS tail,
        Origin || '_' || CAST(CRSDepTime // 100 AS VARCHAR)    AS origin_hr,
        CAST(Month AS INTEGER)                                 AS Month,
        CAST(DayOfWeek AS INTEGER)                             AS DayOfWeek,
        CAST(DayofMonth AS INTEGER)                            AS DayofMonth,
        CAST(DayOfWeek IN (6, 7) AS INTEGER)                   AS is_weekend,
        DAYOFYEAR(FlightDate)                                  AS day_of_year,
        CAST(CRSDepTime // 100 AS INTEGER)                     AS dep_hour,
        CAST(CRSArrTime // 100 AS INTEGER)                     AS arr_hour,
        CAST((CRSDepTime // 100) * 60 + CRSDepTime % 100 AS INTEGER) AS dep_min,
        CAST((CRSArrTime // 100) * 60 + CRSArrTime % 100 AS INTEGER) AS arr_min,
        CRSElapsedTime, Distance
    FROM silver
""")

CAT_COLS = ["Reporting_Airline", "Origin", "Dest", "route", "OriginState", "DestState", "DepTimeBlk"]
NUM_STATIC = ["Distance", "CRSElapsedTime", "dep_min", "arr_min", "Month", "DayOfWeek", "DayofMonth",
              "day_of_year", "is_weekend", "days_to_holiday"]
con.sql("SELECT * FROM base LIMIT 5").df()

,FlightDate,DepDel15,Reporting_Airline,Origin,Dest,OriginState,DestState,DepTimeBlk,route,tail,origin_hr,Month,DayOfWeek,DayofMonth,is_weekend,day_of_year,dep_hour,arr_hour,dep_min,arr_min,CRSElapsedTime,Distance
0,2022-01-14,0,YX,CMH,DCA,OH,VA,1200-1259,CMH-DCA,N119HQ,CMH_12,1,5,14,0,14,12,13,744,832,88.0000,323.0000
1,2022-01-15,0,YX,CMH,DCA,OH,VA,1200-1259,CMH-DCA,N122HQ,CMH_12,1,6,15,1,15,12,13,744,832,88.0000,323.0000
2,2022-01-16,0,YX,CMH,DCA,OH,VA,1200-1259,CMH-DCA,N412YX,CMH_12,1,7,16,1,16,12,13,744,832,88.0000,323.0000
3,2022-01-17,0,YX,CMH,DCA,OH,VA,1200-1259,CMH-DCA,N405YX,CMH_12,1,1,17,0,17,12,13,744,832,88.0000,323.0000
4,2022-01-18,0,YX,CMH,DCA,OH,VA,1200-1259,CMH-DCA,N420YX,CMH_12,1,2,18,0,18,12,13,744,832,88.0000,323.0000


In [7]:
PRIOR = con.sql(f"SELECT AVG({TARGET}) FROM base WHERE FlightDate <= DATE '{TRAIN_END}'").fetchall()[0][0]
HIST_KEYS = {"origin": "Origin", "route": "route", "airline": "Reporting_Airline",
             "tail": "tail", "origin_hr": "origin_hr"}
W = HIST_WINDOW_DAYS

t0 = time.time()
for name, col in HIST_KEYS.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE hist_{name} AS
        WITH daily AS (
            SELECT {col} AS k, FlightDate, COUNT(*) AS n, SUM({TARGET}) AS d
            FROM base WHERE {col} IS NOT NULL GROUP BY 1, 2
        )
        SELECT k, FlightDate,
               COALESCE(SUM(n) OVER w_all, 0) AS n_all, COALESCE(SUM(d) OVER w_all, 0) AS d_all,
               COALESCE(SUM(n) OVER w_win, 0) AS n_win, COALESCE(SUM(d) OVER w_win, 0) AS d_win
        FROM daily
        WINDOW w_all AS (PARTITION BY k ORDER BY FlightDate
                         ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING),
               w_win AS (PARTITION BY k ORDER BY FlightDate
                         RANGE BETWEEN INTERVAL {W} DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
    """)
print(f"Tablas de historial listas en {time.time() - t0:,.0f}s | prior (tasa de entrenamiento) = {PRIOR:.4f}")

HIST_COLS = [f"{k}_rate_{s}" for k in HIST_KEYS for s in ("all", f"{W}d")]

Tablas de historial listas en 59s | prior (tasa de entrenamiento) = 0.2084


In [8]:
t0 = time.time()
con.execute("CREATE OR REPLACE TABLE load_origin     AS SELECT Origin, FlightDate, dep_hour, COUNT(*) AS origin_dep_load FROM base GROUP BY ALL")
con.execute("CREATE OR REPLACE TABLE load_origin_day AS SELECT Origin, FlightDate, COUNT(*) AS origin_day_load FROM base GROUP BY ALL")
con.execute("CREATE OR REPLACE TABLE load_dest       AS SELECT Dest, FlightDate, arr_hour, COUNT(*) AS dest_arr_load FROM base GROUP BY ALL")
con.execute("CREATE OR REPLACE TABLE legs AS SELECT tail, FlightDate, dep_min, arr_min, DepDel15 FROM base WHERE tail IS NOT NULL")
con.execute("""
    CREATE OR REPLACE TABLE ap_cum AS
    WITH g AS (SELECT Origin, FlightDate, dep_min, COUNT(*) AS n, SUM(DepDel15) AS d FROM base GROUP BY ALL)
    SELECT Origin, FlightDate, dep_min, SUM(n) OVER w AS cum_n, SUM(d) OVER w AS cum_d
    FROM g
    WINDOW w AS (PARTITION BY Origin, FlightDate ORDER BY dep_min ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
""")
print(f"Tablas de congestión y estado del día listas en {time.time() - t0:,.0f}s")

SCHED_COLS  = ["origin_dep_load", "origin_day_load", "dest_arr_load", "tail_leg_num", "turnaround_min"]
STATUS_COLS = ["prev_leg_delayed", "tail_delayed_known", "tail_known_legs", "origin_known_today", "origin_delay_rate_today"]

Tablas de congestión y estado del día listas en 39s


## 2. Preparar las muestras originales, calcular características por bloques y guardar Parquet
No cambies `N_TRAIN`, `N_EVAL`, fechas ni horizonte. El tamaño de **bloque** (`CHUNK_ROWS`) no es el tamaño de muestra: solo limita cuántas filas se materializan a la vez en pandas. Si necesitas cambiar `CHUNK_ROWS` tras una interrupción, usa una carpeta de salida nueva para evitar mezclar fragmentos.

In [ ]:
H = PRED_HORIZON_MIN

def build_split(where, n_rows, source_sql=None):
    q = source_sql if source_sql is not None else f"SELECT b.*, c.days_to_holiday FROM base b JOIN calendar c USING (FlightDate) WHERE {where}"
    if n_rows:
        q = f"SELECT * FROM ({q}) USING SAMPLE reservoir({n_rows} ROWS) REPEATABLE ({SEED})"
    con.execute(f"""CREATE OR REPLACE TEMP TABLE smp AS
        SELECT ROW_NUMBER() OVER () AS rid, *, dep_min - {H} - 15 AS cut_min, dep_min - {H} AS cut_arr FROM ({q})""")

    # Aeronave: rotación (solo itinerario) y estado de tramos anteriores conocidos al corte.
    con.execute("""CREATE OR REPLACE TEMP TABLE tail_feat AS
        SELECT s.rid,
               COUNT(l.dep_min)                                          AS tail_prev_legs,
               s.dep_min - arg_max(l.arr_min, l.dep_min)                 AS turnaround_raw,
               SUM(l.DepDel15) FILTER (WHERE l.arr_min <= s.cut_arr)     AS tail_delayed_known,
               COUNT(*)        FILTER (WHERE l.arr_min <= s.cut_arr)     AS tail_known_legs,
               arg_max(l.DepDel15, l.dep_min) FILTER (WHERE l.arr_min <= s.cut_arr) AS prev_leg_delayed
        FROM smp s
        LEFT JOIN legs l ON l.tail = s.tail AND l.FlightDate = s.FlightDate AND l.dep_min < s.dep_min
        WHERE s.tail IS NOT NULL
        GROUP BY s.rid, s.dep_min""")

    # Aeropuerto: vuelos del mismo origen y día ya "resueltos" al corte (ASOF sobre acumulados).
    con.execute("""CREATE OR REPLACE TEMP TABLE ap_feat AS
        SELECT s.rid, a.cum_n AS origin_known_today, a.cum_d AS origin_delayed_today
        FROM smp s ASOF LEFT JOIN ap_cum a
          ON s.Origin = a.Origin AND s.FlightDate = a.FlightDate AND s.cut_min >= a.dep_min""")

    joins, cols = [], []
    for name, col in HIST_KEYS.items():
        joins.append(f"LEFT JOIN hist_{name} h_{name} ON h_{name}.k = s.{col} AND h_{name}.FlightDate = s.FlightDate")
        for suffix, (n, d) in {"all": ("n_all", "d_all"), f"{W}d": ("n_win", "d_win")}.items():
            cols.append(f"(COALESCE(h_{name}.{d}, 0) + {HIST_SMOOTH} * CAST({PRIOR} AS DOUBLE)) / "
                        f"(COALESCE(h_{name}.{n}, 0) + {HIST_SMOOTH}) AS {name}_rate_{suffix}")
    sql = f"""
        SELECT s.* EXCLUDE (cut_min, cut_arr, rid), {', '.join(cols)},
               lo.origin_dep_load, lod.origin_day_load, ld.dest_arr_load,
               tf.tail_prev_legs + 1                                   AS tail_leg_num,
               CASE WHEN tf.turnaround_raw >= 0 THEN tf.turnaround_raw END AS turnaround_min,
               tf.prev_leg_delayed, tf.tail_delayed_known, tf.tail_known_legs,
               COALESCE(af.origin_known_today, 0)                      AS origin_known_today,
               COALESCE(af.origin_delayed_today, 0)                    AS origin_delayed_today
        FROM smp s {' '.join(joins)}
        LEFT JOIN load_origin     lo  ON lo.Origin = s.Origin AND lo.FlightDate = s.FlightDate AND lo.dep_hour = s.dep_hour
        LEFT JOIN load_origin_day lod ON lod.Origin = s.Origin AND lod.FlightDate = s.FlightDate
        LEFT JOIN load_dest       ld  ON ld.Dest = s.Dest AND ld.FlightDate = s.FlightDate AND ld.arr_hour = s.arr_hour
        LEFT JOIN tail_feat tf ON tf.rid = s.rid
        LEFT JOIN ap_feat   af ON af.rid = s.rid"""
    df = con.sql(sql).df()
    # DuckDB devuelve enteros con NULL como tipos anulables de pandas (pd.NA): se pasan a float (NaN).
    for c in NUM_STATIC + SCHED_COLS + STATUS_COLS + ["origin_delayed_today"]:
        if c in df.columns:     # origin_delay_rate_today se calcula justo debajo
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    # Tasa de retraso de HOY en el aeropuerto, suavizada hacia su tasa reciente (sin NULL).
    df["origin_delay_rate_today"] = (df.origin_delayed_today + 10 * df.origin_rate_30d) / (df.origin_known_today + 10)
    return df


# Persistencia por lotes: cambia solo CHUNK_ROWS (no la muestra total).
import gc
import pyarrow.parquet as pq

CHUNK_ROWS = 50_000
PARTS_BASE = Path(f"../data/checkpoints/features_seed{SEED}_h{PRED_HORIZON_MIN}_chunk{CHUNK_ROWS}")
PARTS_BASE.mkdir(parents=True, exist_ok=True)

d0 = pd.Timestamp(d_min) + pd.Timedelta(days=WARMUP_DAYS)
specs = [
    ("train", f"FlightDate <= DATE '{TRAIN_END}' AND FlightDate >= DATE '{d0.date()}'", N_TRAIN),
    ("val", f"FlightDate > DATE '{TRAIN_END}' AND FlightDate <= DATE '{VAL_END}'", N_EVAL),
    ("test", f"FlightDate > DATE '{VAL_END}'", N_EVAL),
]

for split_name, where, requested in specs:
    out_dir = PARTS_BASE / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    done = out_dir / "_SUCCESS"
    if done.exists():
        print(f"✔ {split_name}: ya terminado, se omite")
        continue

    # Conservar la MISMA muestra reservoir a lo largo de todos los bloques.
    sample_table = f"sample_{split_name}_{requested}_s{SEED}"
    exists = con.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_schema='main' AND table_name=?",
        [sample_table]
    ).fetchone()[0]
    if not exists:
        q = f"SELECT b.*, c.days_to_holiday FROM base b JOIN calendar c USING (FlightDate) WHERE {where}"
        con.execute(f"""CREATE TABLE {sample_table} AS
            SELECT ROW_NUMBER() OVER () AS sample_row, ss.*
            FROM (SELECT * FROM ({q}) USING SAMPLE reservoir({requested} ROWS) REPEATABLE ({SEED})) ss""")
    total = con.execute(f"SELECT COUNT(*) FROM {sample_table}").fetchone()[0]
    print(f"{split_name}: muestra fija {total:,} registros; bloques de {CHUNK_ROWS:,}")

    for offset in range(0, total, CHUNK_ROWS):
        part_path = out_dir / f"part_{offset // CHUNK_ROWS:04d}.parquet"
        if part_path.exists():
            print(f"  ya existe {part_path.name}, se omite")
            continue
        source_q = (
            f"SELECT * EXCLUDE (sample_row) FROM {sample_table} "
            f"WHERE sample_row > {offset} AND sample_row <= {offset + CHUNK_ROWS}"
        )
        df = build_split('1=1', None, source_sql=source_q)
        tmp = part_path.with_suffix('.parquet.tmp')
        df.to_parquet(tmp, index=False, compression='zstd')
        tmp.replace(part_path)
        print(f"  guardado {part_path.name}: {len(df):,} filas")
        del df
        gc.collect()

    actual = sum(pq.ParquetFile(p).metadata.num_rows for p in out_dir.glob('part_*.parquet'))
    assert actual == total, f"{split_name}: esperadas {total:,} filas, encontradas {actual:,}"
    done.write_text(f"rows={actual}\nseed={SEED}\nrequested={requested}\nchunk={CHUNK_ROWS}\n")
    con.execute(f"DROP TABLE {sample_table}")
    print(f"✔ {split_name}: completado con {actual:,} filas")

print("\n✔ Etapa A completa. Directorio:", PARTS_BASE.resolve())


train: muestra fija 4,000,000 registros; bloques de 50,000
  guardado part_0000.parquet: 50,000 filas


## 3. Verificación sin cargar los millones de filas en RAM
Usa DuckDB para leer los Parquet por partición. El código siguiente calcula filas y tasa real de retrasos sin crear un DataFrame masivo de pandas.

In [5]:
for split_name, _, expected in specs:
    path = (PARTS_BASE / split_name / "part_*.parquet").as_posix()
    n, rate, dmin, dmax = con.execute(
        f"SELECT COUNT(*), AVG(DepDel15), MIN(FlightDate), MAX(FlightDate) FROM read_parquet('{path}', union_by_name=true)"
    ).fetchone()
    print(f"{split_name:5} | filas={n:,} (solicitadas={expected:,}) | tasa={rate:.3%} | {dmin} → {dmax}")
    assert n == expected, f"{split_name} no contiene el tamaño de muestra esperado"

# Liberar conexión con la base para permitir nuevos procesos / kernels.
con.close()
print('✔ Comprobaciones terminadas y conexión cerrada.')

NameError: name 'specs' is not defined

In [6]:
from pathlib import Path
import pyarrow.parquet as pq

base = Path(
    "/home/ec2-user/SageMaker/data/checkpoints"
)

carpetas = sorted(
    base.glob("features_seed42_h240_chunk*")
)

esperados = {
    "train": 4_000_000,
    "val": 1_000_000,
    "test": 1_000_000
}

if not carpetas:
    print("No se encontraron carpetas de checkpoints.")
else:
    for carpeta in carpetas:
        print(f"\nCARPETA: {carpeta.name}")

        for nombre, esperado in esperados.items():
            directorio = carpeta / nombre
            archivos = sorted(
                directorio.glob("part_*.parquet")
            )

            filas = 0
            for archivo in archivos:
                filas += pq.ParquetFile(
                    archivo
                ).metadata.num_rows

            completo = (directorio / "_SUCCESS").exists()

            print(
                f"{nombre:5} | "
                f"Archivos: {len(archivos):3} | "
                f"Filas: {filas:,} / {esperado:,} | "
                f"SUCCESS: {completo}"
            )

            if completo and filas == esperado:
                print("  ✔ Conjunto completado")
            elif filas > 0:
                print("  ⚠ Revisar: conjunto incompleto o sin confirmar")
            else:
                print("  ✘ Sin archivos")


CARPETA: features_seed42_h240_chunk50000
train | Archivos:  80 | Filas: 4,000,000 / 4,000,000 | SUCCESS: True
  ✔ Conjunto completado
val   | Archivos:  20 | Filas: 1,000,000 / 1,000,000 | SUCCESS: True
  ✔ Conjunto completado
test  | Archivos:  20 | Filas: 1,000,000 / 1,000,000 | SUCCESS: True
  ✔ Conjunto completado


## Siguiente etapa (B)
El notebook original entrenaba cuatro modelos y mantenía sus objetos y matrices simultáneamente. Para ejecutarlo con 16 GiB sin cambiar tamaños ni métricas, hay que **entrenar uno por ejecución**, guardar sus probabilidades de validación/prueba y artefactos y liberar memoria antes de entrenar el siguiente. La etapa C reúne las probabilidades para calcular exactamente las mismas métricas y bootstrap sobre los mismos vuelos.

**No ejecutes el resto del notebook original directamente después de esta etapa**, porque volvería a cargar todo junto. Verifica primero que se completaron los tres conjuntos.
